In [ ]:
import json, re

DATA_PATH = "../Data/data/en-US.jsonl"
SLOT_REGEX = re.compile(r"\[([^\[\]:]+?)\s*:\s*([^\[\]]+?)\]")

def parse_slots(annot_utt):
    """Return list of (slot_type, slot_span) pairs from annot_utt."""
    # get list of (slot_type, slot_span) from annot_utt
    return [(t.strip(), s.strip()) for t, s in SLOT_REGEX.findall(annot_utt)]

def load_massive(path=DATA_PATH):
    splits = {"train": [], "dev": [], "test": []}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            splits[r["partition"]].append({
                "id": r["id"],
                "utt": r["utt"],              # model input
                "intent": r["intent"],        # task 1 label
                "annot_utt": r["annot_utt"],
                "slots": parse_slots(r["annot_utt"]),  # task 2 label
            })
    return splits



In [ ]:
splits = load_massive()
expected = {"train": 11514, "dev": 2033, "test": 2974}
for k, n in expected.items():
    print(f"{k}: {len(splits[k])}")
    assert len(splits[k]) == n
assert sum(len(v) for v in splits.values()) == 16521

intents = sorted({r["intent"] for v in splits.values() for r in v})
slot_types = sorted({t for v in splits.values() for r in v for t, _ in r["slots"]})
print("intents:", len(intents), "| slot types:", len(slot_types))
print(splits["train"][1])